# V6 — Florence-2-base · H1 (Bootstrap Colab T4)

Notebook **base** del hito H1 del plan V6. Solo carga el modelo, el dataset y verifica el smoke test zero-shot. **No entrena** (eso es H3).

Plan: `Documentacion/plan_v6.md`. Decisiones: `memory/bot/decisions.md`.

**Stack** elegido para Florence-2 + T4 16 GB (NO derivado de V5):
- `transformers>=4.41,<4.46` (>=4.46 rompe `trust_remote_code` de Florence-2)
- `accelerate`, `peft`, `einops`, `timm`, `datasets`
- **fp16** (T4 compute 7.5 no soporta bf16 eficiente)
- **Sin Unsloth, sin flash-attn** (T4 no soporta FA2)

**Tarea**: tag custom `<EXTRACT_TOTAL>`. Output esperado tras H3: `total<loc_x1><loc_y1><loc_x2><loc_y2>`.

## A · Check entorno

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free,compute_cap --format=csv,noheader
import torch
print('torch:', torch.__version__, '| cuda:', torch.version.cuda, '| available:', torch.cuda.is_available())
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f'compute capability: {cap}  → fp16 {"sí" if cap[0]<8 else "posible"}, bf16 nativo {"no" if cap[0]<8 else "sí"}')

## B · Instalación de dependencias

NO reinstalar torch (la imagen base de Colab ya trae uno compatible con T4 + CUDA 12). NO instalar flash-attn.

In [ ]:
!pip install -q \
    "transformers>=4.41,<4.46" \
    "accelerate>=0.30" \
    "peft>=0.11" \
    "einops>=0.8" \
    "timm>=0.9" \
    "datasets>=2.19" \
    "huggingface_hub>=0.24" \
    "pillow>=10"

## C · Mount Drive (preparado para checkpoints de H3)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
CKPT_DIR = '/content/drive/MyDrive/TFG/V6_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print('checkpoints futuros →', CKPT_DIR)

## D · Login HF (token desde Colab Secrets)

Crear el secret `HF_TOKEN` en Colab (panel izquierdo → 🔑) con un token **write** del usuario `Lacax`.

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))
print('HF login ok')

## E · Cargar Florence-2-base en fp16

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

MODEL_ID = 'microsoft/Florence-2-base'
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16,
).to('cuda').eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
vram = torch.cuda.memory_allocated() / 1e9
print(f'Florence-2-base loaded · params≈{n_params:.1f}M · fp16 VRAM≈{vram:.2f} GB')

## F · Cargar dataset Lacax/Tickets-total

In [ ]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from PIL import Image
import json, io
import matplotlib.pyplot as plt

REPO = 'Lacax/Tickets-total'

def load_split(split: str):
    """Lee el JSONL del split desde el repo y devuelve list[dict]."""
    path = hf_hub_download(REPO, f'dataset_total_{split}.jsonl', repo_type='dataset')
    return [json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]

def load_image(image_name: str, labeled: bool = False):
    """original/ -> entrada al modelo (sin marcas).
    etiquetadas/ -> solo verificacion visual humana (NO usar en training: data leakage)."""
    folder = 'etiquetadas' if labeled else 'original'
    path = hf_hub_download(REPO, f'{folder}/{image_name}', repo_type='dataset')
    return Image.open(path).convert('RGB')

train = load_split('train')
val   = load_split('val')
test  = load_split('test')
print(f'splits -> train={len(train)}  val={len(val)}  test={len(test)}')
print('ejemplo val[0]:', val[0])

# Verificacion visual: original (lo que ve el modelo) vs etiquetada (con bbox rojo)
sample = val[0]
fig, axes = plt.subplots(1, 2, figsize=(10, 7))
axes[0].imshow(load_image(sample['image_path'], labeled=False))
axes[0].set_title('original (input training)'); axes[0].axis('off')
axes[1].imshow(load_image(sample['image_path'], labeled=True))
axes[1].set_title(f"etiquetada (verificacion) - total={sample['total']}"); axes[1].axis('off')
plt.tight_layout(); plt.show()


## G · Añadir tag custom `<EXTRACT_TOTAL>`

Florence-2 admite tags custom siempre que se redimensione el embedding. La tarea se aprenderá en H3 desde cero, pero el setup vive aquí.

In [ ]:
EXTRACT_TAG = '<EXTRACT_TOTAL>'
added = processor.tokenizer.add_tokens([EXTRACT_TAG], special_tokens=True)
print(f'añadidos {added} tokens nuevos (0 si ya existía)')
model.resize_token_embeddings(len(processor.tokenizer))
tag_ids = processor.tokenizer(EXTRACT_TAG, add_special_tokens=False).input_ids
print(f"Tag '{EXTRACT_TAG}' tokenized to ids={tag_ids}  (debería ser 1 id único)")

## H · Smoke test zero-shot

Sin entrenar el tag, la salida será ruidosa (no vamos a evaluar correctitud). **El criterio es no-OOM y < 12 GB de VRAM** para dejar margen al training en H3.

In [ ]:
sample = val[0]
img = load_image(sample['image_path'])  # labeled=False -> original sin marcas
print('imagen:', sample['image_path'], 'size:', img.size, 'total GT:', sample['total'])

inputs = processor(text=EXTRACT_TAG, images=img, return_tensors='pt').to('cuda', torch.float16)
with torch.inference_mode():
    out = model.generate(
        input_ids=inputs['input_ids'],
        pixel_values=inputs['pixel_values'],
        max_new_tokens=20,
        do_sample=False,
        num_beams=1,
    )
decoded = processor.batch_decode(out, skip_special_tokens=False)[0]
print('output (zero-shot, ruidoso esperado):', repr(decoded))

vram = torch.cuda.memory_allocated() / 1e9
peak = torch.cuda.max_memory_allocated() / 1e9
print(f'VRAM tras smoke test: actual={vram:.2f} GB - pico={peak:.2f} GB  (objetivo < 12 GB)')


---

**H1 cerrado** si:
- Celda E imprime `params≈230M`
- Celda G imprime `tokenized to ids=[ÚNICO_ID]`
- Celda H ejecuta sin OOM y reporta pico < 12 GB.

Siguiente paso (H2): `DataCollator` que mapee `{image, total, bbox}` → `(pixel_values, input_ids=tag, labels=total<loc>)`.